# 04 — Explainability, Ethical AI & Bias Auditing

Covers capstone Step 5: SHAP (global + local) on the chosen XGBoost ranker,
PDP/ICE, a LIME cross-check, and the pinned fairness protocol — customer region
as the socioeconomic proxy, evaluated on the REAL serving population (each
holdout customer's Stage-1 candidate list, never per-user random negatives),
separation-style metrics first, n + bootstrap CIs in every group table,
suppression below n=30, and matched mitigations measured before/after on the
same metric. Provider-side exposure (Gini/Lorenz, long-tail, seller size) gets
the MMR popularity-penalty re-ranker and the fairness-accuracy trade-off chart.

In [1]:
import json
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.evaluate import bootstrap_ci, rank_metrics
from src.fairness import (exposure_gini, group_metric_table, lorenz_points,
                          rerank_with_popularity_penalty)
from src.features import interactions, sample_negatives
from src.models.candidate_gen import top_k_from_scores
from src.pipeline import build_artifacts, candidate_frame, ranker_matrix

sns.set_theme(style="whitegrid")
SEED = 42
FIG_DIR = REPO_ROOT / "reports" / "figures"
MODELS_DIR = REPO_ROOT / "models"

art = build_artifacts()
LW, HO = art["windows"]["label_window"], art["windows"]["holdout"]
xgb = joblib.load(MODELS_DIR / "ranker_xgboost.joblib")
metrics = json.loads((MODELS_DIR / "metrics.json").read_text())
TAU = metrics["rankers"]["xgboost"]["tau"]
print(f"artefacts rebuilt; XGBoost tau = {TAU:.4f} (selected out-of-fold in notebook 03)")

artefacts rebuilt; XGBoost tau = 0.2720 (selected out-of-fold in notebook 03)


In [2]:
# Training pairs (same protocol as notebook 03: half hard negatives from the
# customer's own candidate list, half random, seed 42) - needed for the
# ThresholdOptimizer fit and the unawareness retrain
cfg_ns = art["chosen"]["negative_sampling"]
rng = np.random.default_rng(cfg_ns["seed"])
pos = interactions(art["delivered"], art["items"], *LW)[
    ["customer_unique_id", "product_id"]].drop_duplicates()
pos_sets = pos.groupby("customer_unique_id")["product_id"].agg(set)
n_hard = int(cfg_ns["ratio"] * cfg_ns["hard_share"])
hard_rows = []
for cust, bought in pos_sets.items():
    cands, _ = art["router"].recommend(cust, region=art["geo_region"].get(cust), k=None)
    pool = [p for p in art["im_fw"].product_ids[cands] if p not in bought]
    take = rng.choice(len(pool), size=min(n_hard * len(bought), len(pool)),
                      replace=False) if pool else []
    hard_rows += [(cust, pool[i]) for i in take]
hard = pd.DataFrame(hard_rows, columns=["customer_unique_id", "product_id"])
rand = sample_negatives(pos, art["products"]["product_id"].to_numpy(),
                        ratio=cfg_ns["ratio"] - n_hard, seed=cfg_ns["seed"])
train_pairs = pd.concat([pos.assign(y=1), hard.assign(y=0), rand.assign(y=0)],
                        ignore_index=True).drop_duplicates(
    ["customer_unique_id", "product_id"], keep="first")
X_tr = ranker_matrix(train_pairs, art)
y_tr = train_pairs["y"].to_numpy()
reg_tr = train_pairs["customer_unique_id"].map(art["geo_region"]).fillna("unknown")
FEATS = list(X_tr.columns)
(MODELS_DIR / "feature_columns.json").write_text(json.dumps(FEATS))
print(f"train pairs {X_tr.shape}; regions: {reg_tr.value_counts().to_dict()}")

train pairs (106293, 32); regions: {'Southeast': 74446, 'South': 14899, 'Northeast': 9243, 'Center-West': 6030, 'North': 1675}


In [3]:
# Evaluation population: every holdout customer's Stage-1 candidate list,
# labelled by what they actually bought. Never per-user random negatives -
# a fixed sampling ratio would force every region's base rate to the ratio
# and make demographic parity trivially, meaninglessly 'fair'.
ho_pos = interactions(art["delivered"], art["items"], *HO)[
    ["customer_unique_id", "product_id"]].drop_duplicates()
ho_sets = ho_pos.groupby("customer_unique_id")["product_id"].agg(set)
cands_df, routes = candidate_frame(art, ho_sets.index)
key = pd.MultiIndex.from_frame(ho_pos)
cand_key = pd.MultiIndex.from_frame(cands_df[["customer_unique_id", "product_id"]])
y_cand = cand_key.isin(key).astype(int)
X_cand = ranker_matrix(cands_df, art, columns=FEATS)
p_cand = xgb.predict_proba(X_cand)[:, 1]
reg_cand = cands_df["customer_unique_id"].map(art["geo_region"]).fillna("unknown").to_numpy()

base = pd.DataFrame({"y": y_cand, "region": reg_cand}).groupby("region")["y"].agg(
    ["count", "sum", "mean"]).rename(columns={"count": "n_pairs", "sum": "n_pos",
                                              "mean": "base_rate"})
print(f"candidate pairs: {len(cands_df)} from {cands_df['customer_unique_id'].nunique()} customers")
print("per-region base rates on the candidate population "
      "(printed next to every fairness table, per the protocol):")
print(base.round(4).to_string())

candidate pairs: 919500 from 18390 customers
per-region base rates on the candidate population (printed next to every fairness table, per the protocol):
             n_pairs  n_pos  base_rate
region                                
Center-West    52950     30     0.0006
North          15000     10     0.0007
Northeast      81000     72     0.0009
South         121100     64     0.0005
Southeast     649450    426     0.0007


## Explainability — SHAP, PDP/ICE, LIME

Global attribution on a serving-population sample, then two local waterfalls:
one candidate served to a warm customer, one to a cold customer. Remember the
Step 3 warning: importance on a cold-fill-dominated feature is really
importance of *being cold*.

In [4]:
import shap

rng2 = np.random.default_rng(SEED)
samp = rng2.choice(len(X_cand), size=4000, replace=False)
X_shap = X_cand.iloc[samp]
explainer = shap.TreeExplainer(xgb)
sv = explainer(X_shap)

fig = plt.figure(figsize=(9, 7))
shap.plots.beeswarm(sv, max_display=15, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.close("all")

fig = plt.figure(figsize=(7, 6))
shap.plots.bar(sv, max_display=15, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_shap_bar.png", dpi=150, bbox_inches="tight")
plt.close("all")

mean_abs = pd.Series(np.abs(sv.values).mean(axis=0), index=FEATS).sort_values(ascending=False)
print("mean |SHAP| top 10:")
print(mean_abs.head(10).round(4).to_string())

C:\Users\jmonz\Documents\GitHub\ecommerce-recommender-capstone\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


mean |SHAP| top 10:
p_popularity                    1.4918
p_review_mean                   0.2615
s_seller_order_count            0.2251
p_product_weight_g              0.2005
s_seller_review_mean            0.1655
distance_km                     0.1511
p_price_band                    0.1460
p_median_price_w                0.1332
p_category_satisfaction_rate    0.1293
p_product_description_lenght    0.1275


In [5]:
# Two local explanations: the top-scored candidate for a warm customer and
# for a cold customer ("why is this product on this person's list?")
warm_custs = routes[routes == "hybrid_only"].index
cold_custs = routes[routes == "regional_popularity"].index
cases = {}
for label, cust in [("warm", warm_custs[0]), ("cold", cold_custs[0])]:
    mask = cands_df["customer_unique_id"] == cust
    i = int(np.argmax(np.where(mask, p_cand, -1)))
    cases[label] = i
    sv_i = explainer(X_cand.iloc[[i]])
    fig = plt.figure(figsize=(8, 6))
    shap.plots.waterfall(sv_i[0], max_display=12, show=False)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"04_shap_waterfall_{label}.png", dpi=150, bbox_inches="tight")
    plt.close("all")
    print(f"{label}: customer {cust[:8]}..., product {cands_df.iloc[i]['product_id'][:8]}..., "
          f"score {p_cand[i]:.3f}, route {routes[cust]}")

warm: customer 0178b244..., product 99a4788c..., score 0.864, route hybrid_only


cold: customer 000e3092..., product aca2eb7d..., score 0.327, route regional_popularity


In [6]:
from sklearn.inspection import PartialDependenceDisplay

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
PartialDependenceDisplay.from_estimator(
    xgb, X_shap, ["p_popularity", "price_delta", "distance_km"],
    kind="both", subsample=200, random_state=SEED, ax=ax,
)
fig.suptitle("PDP + ICE on the XGBoost ranker (200 ICE lines)")
fig.tight_layout()
fig.savefig(FIG_DIR / "04_pdp_ice.png", dpi=150)
plt.close(fig)
print("PDP/ICE saved")

PDP/ICE saved


In [7]:
from lime.lime_tabular import LimeTabularExplainer

lime_exp = LimeTabularExplainer(
    X_tr.to_numpy(dtype=float), feature_names=FEATS, class_names=["skip", "buy"],
    mode="classification", discretize_continuous=True, random_state=SEED,
)
for label, i in cases.items():
    exp = lime_exp.explain_instance(X_cand.iloc[i].to_numpy(dtype=float),
                                    xgb.predict_proba, num_features=8)
    print(f"\nLIME ({label} case):")
    for feat, wgt in exp.as_list():
        print(f"  {feat:45s} {wgt:+.4f}")


LIME (warm case):
  p_popularity > 81.00                          -0.1464
  cf_signal <= 0.00                             -0.1280
  category_match > 0.00                         -0.0934
  s_seller_order_count > 474.00                 +0.0523
  price_delta <= 0.00                           -0.0488
  0.00 < region_Southeast <= 1.00               +0.0485
  c_frequency > 0.00                            +0.0467
  has_cf_signal <= 0.00                         -0.0346

LIME (cold case):
  p_popularity > 81.00                          -0.1442
  category_match <= 0.00                        +0.1227
  has_cf_signal <= 0.00                         +0.0649
  region_Southeast <= 0.00                      -0.0512
  price_delta <= 0.00                           -0.0478
  s_seller_order_count > 474.00                 +0.0464
  c_monetary <= 0.00                            -0.0397
  region_Center-West > 0.00                     +0.0359


## Fairness audit — region as the socioeconomic proxy

Decision rule: predict "recommend-worthy" iff score >= tau (tau fixed
out-of-fold in notebook 03). Primary criteria are separation-style (per-region
TPR on real purchases, per-region AUC); demographic parity difference and
disparate impact ratio are reported descriptively because regional base rates
genuinely differ. Groups below n=30 are suppressed; TPR/AUC need 20+ positives;
the pre-registered fallback merges North into Northeast if North lacks support.

In [8]:
table5 = group_metric_table(y_cand, p_cand, reg_cand, TAU)
print("five-region audit (candidate population):")
print(table5.round(4).to_string())
print(f"\nDP difference (descriptive): {table5.attrs['dp_difference']:.4f}")
print(f"DI ratio (descriptive):      {table5.attrs['di_ratio']:.4f}")
print(f"TPR gap (primary):           {table5.attrs['tpr_gap']:.4f}")

if table5["tpr"].isna().any():
    reg_merged = np.where(np.isin(reg_cand, ["North", "Northeast"]), "N+NE", reg_cand)
    table_m = group_metric_table(y_cand, p_cand, reg_merged, TAU)
    print("\npre-registered fallback (North merged into Northeast):")
    print(table_m.round(4).to_string())
    print(f"TPR gap (merged):            {table_m.attrs['tpr_gap']:.4f}")

five-region audit (candidate population):
                  n  n_pos  base_rate  selection_rate     tpr  tpr_lo  tpr_hi     auc
group                                                                                
Center-West   52950     30     0.0006          0.0246  0.1000  0.0000  0.2333  0.6786
North         15000     10     0.0007          0.0205     NaN     NaN     NaN     NaN
Northeast     81000     72     0.0009          0.0249  0.2222  0.1389  0.3194  0.8246
South        121100     64     0.0005          0.0117  0.0312  0.0000  0.0781  0.7095
Southeast    649450    426     0.0007          0.0073  0.0282  0.0141  0.0446  0.7137

DP difference (descriptive): 0.0176
DI ratio (descriptive):      0.2944
TPR gap (primary):           0.1941



pre-registered fallback (North merged into Northeast):
                  n  n_pos  base_rate  selection_rate     tpr  tpr_lo  tpr_hi     auc
group                                                                                
Center-West   52950     30     0.0006          0.0246  0.1000  0.0000  0.2333  0.6786
N+NE          96000     82     0.0009          0.0242  0.2073  0.1220  0.3049  0.8038
South        121100     64     0.0005          0.0117  0.0312  0.0000  0.0781  0.7095
Southeast    649450    426     0.0007          0.0073  0.0282  0.0141  0.0446  0.7137
TPR gap (merged):            0.1791


In [9]:
# Fairness through unawareness check: retrain the same XGBoost without
# region-derived features. Expected (and instructive) result: gaps barely move,
# because freight/price features encode geography anyway.
from xgboost import XGBClassifier

GEO_FEATS = [c for c in FEATS if c.startswith("region_")] + ["same_state", "distance_km"]
FEATS_NOGEO = [c for c in FEATS if c not in GEO_FEATS]
xgb_nogeo = XGBClassifier(**{k: v for k, v in xgb.get_params().items() if v is not None})
xgb_nogeo.fit(X_tr[FEATS_NOGEO], y_tr)
p_nogeo = xgb_nogeo.predict_proba(X_cand[FEATS_NOGEO])[:, 1]

from sklearn.metrics import roc_auc_score
t_nogeo = group_metric_table(y_cand, p_nogeo, reg_cand, TAU)
print(f"AUC with geography:    {roc_auc_score(y_cand, p_cand):.4f}")
print(f"AUC without geography: {roc_auc_score(y_cand, p_nogeo):.4f}")
print(f"TPR gap with geography:    {table5.attrs['tpr_gap']:.4f}")
print(f"TPR gap without geography: {t_nogeo.attrs['tpr_gap']:.4f}")
print(t_nogeo[["n", "n_pos", "tpr", "selection_rate"]].round(4).to_string())

AUC with geography:    0.7231
AUC without geography: 0.7128
TPR gap with geography:    0.1941
TPR gap without geography: 0.2917
                  n  n_pos     tpr  selection_rate
group                                             
Center-West   52950     30  0.0000          0.0412
North         15000     10     NaN          0.2188
Northeast     81000     72  0.2917          0.0405
South        121100     64  0.0156          0.0019
Southeast    649450    426  0.0117          0.0015


In [10]:
# Matched mitigation 1: region gaps -> fairlearn ThresholdOptimizer
# (group-specific thresholds, TPR parity), before/after on the SAME metrics
from fairlearn.postprocessing import ThresholdOptimizer

to = ThresholdOptimizer(estimator=xgb, constraints="true_positive_rate_parity",
                        objective="balanced_accuracy_score", prefit=True,
                        predict_method="predict_proba")
to.fit(X_tr, y_tr, sensitive_features=reg_tr)
yhat_to = to.predict(X_cand, sensitive_features=pd.Series(reg_cand), random_state=SEED)

after = pd.DataFrame({"y": y_cand, "yhat": yhat_to, "region": reg_cand})
rows = []
for g, grp in after.groupby("region"):
    pos = grp[grp["y"] == 1]
    rows.append({"group": g, "n": len(grp),
                 "tpr_after": pos["yhat"].mean() if len(pos) >= 20 else np.nan,
                 "selection_after": grp["yhat"].mean()})
to_table = pd.DataFrame(rows).set_index("group").join(
    table5[["tpr", "selection_rate"]].rename(columns={"tpr": "tpr_before",
                                                      "selection_rate": "selection_before"}))
print(to_table[["n", "tpr_before", "tpr_after", "selection_before", "selection_after"]]
      .round(4).to_string())
tpr_a = to_table["tpr_after"].dropna()
print(f"\nTPR gap: before {table5.attrs['tpr_gap']:.4f} -> after "
      f"{(tpr_a.max() - tpr_a.min()):.4f}")
from sklearn.metrics import balanced_accuracy_score
print(f"balanced accuracy cost: {balanced_accuracy_score(y_cand, p_cand >= TAU):.4f} -> "
      f"{balanced_accuracy_score(y_cand, yhat_to):.4f}")

C:\Users\jmonz\Documents\GitHub\ecommerce-recommender-capstone\.venv\Lib\site-packages\fairlearn\postprocessing\_interpolated_thresholder.py:149: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  positive_probs[sensitive_feature_vector == a] = interpolated_predictions[


                  n  tpr_before  tpr_after  selection_before  selection_after
group                                                                        
Center-West   52950      0.1000     0.1000            0.0246           0.0361
North         15000         NaN        NaN            0.0205           0.0145
Northeast     81000      0.2222     0.2917            0.0249           0.0325
South        121100      0.0312     0.0312            0.0117           0.0109
Southeast    649450      0.0282     0.0399            0.0073           0.0101

TPR gap: before 0.1941 -> after 0.2604
balanced accuracy cost: 0.5229 -> 0.5297


## Provider-side exposure & the popularity-penalty mitigation

The final pipeline's top-10 lists, audited for exposure concentration
(Gini/Lorenz), long-tail share, and small-seller share — then the MMR-style
popularity penalty swept over lambda for the fairness-accuracy trade-off chart.

In [11]:
scored = cands_df.copy()
scored["p"] = p_cand
pop_rank = pd.Series(art["pop_fw"].scores_).rank(pct=True).to_numpy()
pid_to_pct = dict(zip(art["im_fw"].product_ids, pop_rank))
scored["pop_pct"] = scored["product_id"].map(pid_to_pct).fillna(0.0)

item_index = art["im_fw"].item_index
pop_scores = art["pop_fw"].scores_
sellers_small = (art["sell_fw"]["seller_order_count"]
                 < art["sell_fw"]["seller_order_count"].median())
prod_seller = art["prod_fw"]["main_seller_id"]

def audit_lists(topk_by_cust):
    stacked = np.concatenate(list(topk_by_cust.values()))
    idx = item_index.get_indexer(stacked)
    counts = np.bincount(idx[idx >= 0], minlength=len(item_index))
    hits = [rank_metrics(np.asarray(v), ho_sets[c], k_levels=(10,))["hit@10"]
            for c, v in topk_by_cust.items()]
    ndcgs = [rank_metrics(np.asarray(v), ho_sets[c], k_levels=(10,))["ndcg@10"]
             for c, v in topk_by_cust.items()]
    top_decile = np.quantile(pop_scores[pop_scores > 0], 0.9)
    longtail = float(np.mean(pop_scores[idx[idx >= 0]] < max(top_decile, 1)))
    small = prod_seller.reindex(stacked).map(sellers_small).eq(True)
    return {"hit@10": float(np.mean(hits)), "ndcg@10": float(np.mean(ndcgs)),
            "gini": exposure_gini(counts), "longtail_share": longtail,
            "coverage": float((counts > 0).sum() / len(counts)),
            "small_seller_share": float(small.mean()),
            "_counts": counts, "_hits": np.array(hits)}

base_lists = rerank_with_popularity_penalty(scored, lam=0.0)
audit0 = audit_lists(base_lists)
print("unmitigated final pipeline (two-stage XGBoost):")
print({k: round(v, 4) for k, v in audit0.items() if not k.startswith("_")})
print(f"catalogue share of small-seller products: "
      f"{prod_seller.map(sellers_small).eq(True).mean():.4f}")

unmitigated final pipeline (two-stage XGBoost):
{'hit@10': 0.0146, 'ndcg@10': 0.0069, 'gini': 0.9986, 'longtail_share': 0.0101, 'coverage': 0.0706, 'small_seller_share': 0.0002}
catalogue share of small-seller products: 0.0588


In [12]:
lams = [0.0, 0.02, 0.05, 0.1, 0.2, 0.4]
sweep = {}
for lam in lams:
    sweep[lam] = audit_lists(rerank_with_popularity_penalty(scored, lam=lam))
sweep_df = pd.DataFrame({lam: {k: v for k, v in a.items() if not k.startswith("_")}
                         for lam, a in sweep.items()}).T
print(sweep_df.round(4).to_string())

# chosen lambda: the largest whose hit@10 stays within 5% relative of unmitigated
ok = [lam for lam in lams if sweep[lam]["hit@10"] >= 0.95 * sweep[0.0]["hit@10"]]
LAM = max(ok)
print(f"\nchosen lambda = {LAM} (largest with <=5% relative hit@10 cost)")

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(lams, [sweep[la]["hit@10"] for la in lams], marker="o", color="tab:blue",
         label="HitRate@10")
ax1.set_xlabel("popularity penalty lambda")
ax1.set_ylabel("HitRate@10", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(lams, [sweep[la]["gini"] for la in lams], marker="s", color="tab:red",
         label="exposure Gini")
ax2.plot(lams, [sweep[la]["longtail_share"] for la in lams], marker="^",
         color="tab:orange", label="long-tail share")
ax2.set_ylabel("exposure metrics", color="tab:red")
ax1.axvline(LAM, ls="--", c="grey")
fig.legend(loc="center right")
ax1.set_title("Fairness-accuracy trade-off: popularity penalty sweep")
fig.tight_layout()
fig.savefig(FIG_DIR / "04_tradeoff_sweep.png", dpi=150)
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 5))
for lam, style in [(0.0, "-"), (LAM, "--")]:
    xs, ys = lorenz_points(sweep[lam]["_counts"])
    ax.plot(xs, ys, style, label=f"lambda={lam} (Gini {sweep[lam]['gini']:.3f})")
ax.plot([0, 1], [0, 1], ":", c="grey", label="perfect equality")
ax.set_xlabel("share of catalogue")
ax.set_ylabel("share of top-10 exposure")
ax.legend()
ax.set_title("Lorenz curves of recommendation exposure")
fig.tight_layout()
fig.savefig(FIG_DIR / "04_lorenz.png", dpi=150)
plt.close(fig)
print("figures saved")

      hit@10  ndcg@10    gini  longtail_share  coverage  small_seller_share
0.00  0.0146   0.0069  0.9986          0.0101    0.0706              0.0002
0.02  0.0146   0.0068  0.9986          0.0103    0.0708              0.0002
0.05  0.0146   0.0068  0.9986          0.0106    0.0709              0.0002
0.10  0.0146   0.0068  0.9986          0.0110    0.0706              0.0002
0.20  0.0146   0.0068  0.9986          0.0117    0.0706              0.0001
0.40  0.0144   0.0067  0.9986          0.0128    0.0707              0.0001

chosen lambda = 0.4 (largest with <=5% relative hit@10 cost)


figures saved


### The sweep locates the bias — and the matched mitigation moves upstream

The lambda sweep barely moves Gini or long-tail share. That's not a failed
experiment, it's the audit's sharpest finding: candidates for the 98.5% cold
majority are the regional TOP-50 popularity list, so every candidate is already
a head item (mean popularity percentile ~0.99) and no within-list re-ranking
can reach the tail. Exposure concentration is decided at candidate generation,
so the matched mitigation goes there: reserve 15 of the 50 cold-route slots for
"exploration" items sampled from regional popularity ranks 51-500, then re-rank
with the same XGBoost and measure what it costs.

In [13]:
from src.recommend import regional_popularity

rng3 = np.random.default_rng(SEED)
reg_top500 = regional_popularity(art["fw_inter"], art["im_fw"].item_index, k=500)
N_HEAD, N_EXPLORE = 35, 15

expl_rows = []
for cust in ho_sets.index:
    if routes[cust] == "regional_popularity":
        reg = art["geo_region"].get(cust)
        top = reg_top500.get(reg)
        if top is not None and len(top) > N_HEAD + N_EXPLORE:
            pool = top[N_HEAD:]
            pick = pool[rng3.choice(len(pool), size=N_EXPLORE, replace=False)]
            cand_idx = np.concatenate([top[:N_HEAD], pick])
            expl_rows.append(pd.DataFrame({
                "customer_unique_id": cust,
                "product_id": art["im_fw"].product_ids[cand_idx],
            }))
            continue
    g = cands_df[cands_df["customer_unique_id"] == cust]
    expl_rows.append(g[["customer_unique_id", "product_id"]])
expl_cands = pd.concat(expl_rows, ignore_index=True)

X_expl = ranker_matrix(expl_cands, art, columns=FEATS)
expl_scored = expl_cands.copy()
expl_scored["p"] = xgb.predict_proba(X_expl)[:, 1]
expl_scored["pop_pct"] = expl_scored["product_id"].map(pid_to_pct).fillna(0.0)
expl_lists = rerank_with_popularity_penalty(expl_scored, lam=0.0)
audit_expl = audit_lists(expl_lists)

cmp_mit = pd.DataFrame({
    "unmitigated": {k: v for k, v in audit0.items() if not k.startswith("_")},
    "exploration_slots": {k: v for k, v in audit_expl.items() if not k.startswith("_")},
}).T
print(cmp_mit.round(4).to_string())
rel = (audit_expl["hit@10"] / audit0["hit@10"] - 1) * 100
print(f"\nhit@10 cost: {rel:+.1f}% relative | coverage "
      f"{audit0['coverage']:.3f} -> {audit_expl['coverage']:.3f}, "
      f"long-tail {audit0['longtail_share']:.3f} -> {audit_expl['longtail_share']:.3f}, "
      f"Gini {audit0['gini']:.4f} -> {audit_expl['gini']:.4f}")

                   hit@10  ndcg@10    gini  longtail_share  coverage  small_seller_share
unmitigated        0.0146   0.0069  0.9986          0.0101    0.0706              0.0002
exploration_slots  0.0029   0.0015  0.9849          0.0268    0.0944              0.0012

hit@10 cost: -80.3% relative | coverage 0.071 -> 0.094, long-tail 0.010 -> 0.027, Gini 0.9986 -> 0.9849


### Why it backfired — and the mitigation that survives the diagnosis

The candidate-stage mitigation cost 80% of the hits for modest exposure gains,
and this notebook's own explainability outputs say why: the L1 coefficients and
LIME both give `p_popularity` a NEGATIVE weight. Hard negatives are drawn from
popular candidate lists, so within a candidate set the ranker learned
"popular = probably not purchased" — harmless when every candidate is a head
item, catastrophic when tail items enter the pool, because the ranker actively
promotes them. Exploration therefore has to bypass the ranker: keep the top-7
of the unmitigated list and reserve the last 3 slots for seeded regional
exploration items injected after ranking. Exposure is guaranteed; the hit cost
is bounded by whatever hits sat at ranks 8-10.

In [14]:
rng4 = np.random.default_rng(SEED)
inj_lists = {}
for cust, top in base_lists.items():
    if routes.get(cust) == "regional_popularity":
        reg = art["geo_region"].get(cust)
        top500 = reg_top500.get(reg)
        if top500 is not None and len(top500) > 50:
            pool = art["im_fw"].product_ids[top500[N_HEAD:]]
            picks = rng4.choice(pool, size=3, replace=False)
            keep = [p for p in top if p not in set(picks)][:7]
            inj_lists[cust] = np.concatenate([np.asarray(keep), picks])
            continue
    inj_lists[cust] = np.asarray(top)
audit_inj = audit_lists(inj_lists)

cmp3 = pd.DataFrame({
    "unmitigated": {k: v for k, v in audit0.items() if not k.startswith("_")},
    "exploration_candidates": {k: v for k, v in audit_expl.items() if not k.startswith("_")},
    "reserved_slot_injection": {k: v for k, v in audit_inj.items() if not k.startswith("_")},
}).T
print(cmp3.round(4).to_string())
rel_inj = (audit_inj["hit@10"] / audit0["hit@10"] - 1) * 100
print(f"\nreserved slots: hit@10 cost {rel_inj:+.1f}% relative | coverage "
      f"{audit0['coverage']:.3f} -> {audit_inj['coverage']:.3f}, long-tail "
      f"{audit0['longtail_share']:.3f} -> {audit_inj['longtail_share']:.3f}, "
      f"Gini {audit0['gini']:.4f} -> {audit_inj['gini']:.4f}, small-seller "
      f"{audit0['small_seller_share']:.4f} -> {audit_inj['small_seller_share']:.4f}")

                         hit@10  ndcg@10    gini  longtail_share  coverage  small_seller_share
unmitigated              0.0146   0.0069  0.9986          0.0101    0.0706              0.0002
exploration_candidates   0.0029   0.0015  0.9849          0.0268    0.0944              0.0012
reserved_slot_injection  0.0126   0.0062  0.9921          0.0210    0.1026              0.0011

reserved slots: hit@10 cost -13.8% relative | coverage 0.071 -> 0.103, long-tail 0.010 -> 0.021, Gini 0.9986 -> 0.9921, small-seller 0.0002 -> 0.0011


In [15]:
# Final-list audit per region, before vs after the surviving mitigation
mit_lists = inj_lists
cust_region = pd.Series({c: art["geo_region"].get(c, "unknown") for c in ho_sets.index})

def per_region(topk_by_cust, audit):
    hits = pd.Series(audit["_hits"], index=list(topk_by_cust.keys()))
    rows = []
    for g in sorted(cust_region.unique()):
        custs = cust_region[cust_region == g].index
        h = hits.reindex(custs).dropna()
        stacked = np.concatenate([topk_by_cust[c] for c in custs if c in topk_by_cust])
        pcts = pd.Series(stacked).map(pid_to_pct)
        rows.append({"region": g, "n_customers": len(custs),
                     "hit@10": h.mean(), "mean_pop_pct": float(pcts.mean())})
    return pd.DataFrame(rows).set_index("region")

before_r = per_region(base_lists, audit0)
after_r = per_region(mit_lists, audit_inj)
cmp = before_r.join(after_r, lsuffix="_before", rsuffix="_after")
print(cmp.round(4).to_string())

fairness_out = {
    "tau": TAU, "lambda": LAM,
    "region_audit_5": table5.reset_index().to_dict(orient="records"),
    "dp_difference": table5.attrs["dp_difference"],
    "di_ratio": table5.attrs["di_ratio"],
    "tpr_gap": table5.attrs["tpr_gap"],
    "threshold_optimizer": to_table.reset_index().replace({np.nan: None}).to_dict(orient="records"),
    "exposure_sweep": {str(k): {kk: vv for kk, vv in v.items() if not kk.startswith("_")}
                       for k, v in sweep.items()},
    "exploration_mitigation": {k: v for k, v in audit_expl.items() if not k.startswith("_")},
    "reserved_slot_injection": {k: v for k, v in audit_inj.items() if not k.startswith("_")},
    "final_list_audit": cmp.reset_index().to_dict(orient="records"),
}
(MODELS_DIR / "fairness_metrics.json").write_text(json.dumps(fairness_out, indent=2, default=float))
print("wrote models/fairness_metrics.json")

             n_customers_before  hit@10_before  mean_pop_pct_before  n_customers_after  hit@10_after  mean_pop_pct_after
region                                                                                                                  
Center-West                1059         0.0132               0.9922               1059        0.0123              0.9746
North                       300         0.0133               0.9893                300        0.0100              0.9460
Northeast                  1620         0.0309               0.9961               1620        0.0278              0.9860
South                      2422         0.0103               0.9927               2422        0.0054              0.9864
Southeast                 12989         0.0135               0.9932              12989        0.0122              0.9907
wrote models/fairness_metrics.json


## Takeaways

- **SHAP:** popularity dominates global attribution (mean |SHAP| 1.49 vs 0.26
  for the runner-up), and its DIRECTION is the notebook's key diagnostic - the
  hard-negative protocol taught the ranker that, within a candidate list,
  popular items are less likely to be the purchase. LIME agrees with SHAP's
  top features on both local cases.
- **Region audit (candidate population, per-region base rates printed):**
  TPR@tau ranges from 0.222 (Northeast) to 0.028 (Southeast) - gap 0.194,
  0.179 after the pre-registered N+NE merge (North suppressed at 10
  positives). Per-region AUC: 0.825 (NE) to 0.679 (CW). The disparity FAVOURS
  the Northeast; the final-list audit shows the same (hit@10 0.031 vs
  0.010-0.014 elsewhere). DP difference 0.018 / DI ratio 0.294 are reported
  descriptively - base rates differ across regions by construction.
- **Fairness through unawareness fails, as expected:** dropping the
  geography-derived features costs AUC (0.723 -> 0.713) and WIDENS the TPR gap
  (0.194 -> 0.292). Freight and price encode geography anyway.
- **ThresholdOptimizer is an honest negative:** group thresholds fitted on the
  label window did not transfer to the holdout's tiny positive counts (gap
  point estimate 0.194 -> 0.260, both within the groups' wide CIs).
- **Provider-side exposure is extreme:** Gini 0.9986, 7.1% catalogue coverage,
  small sellers get 0.02% of slots vs 5.9% of the catalogue.
- **The lambda-penalty sweep is inert (Gini unchanged at every lambda) - and
  that locates the bias at candidate generation:** every cold-route candidate
  is already a head item, so no within-list re-ranking can reach the tail.
- **The candidate-stage mitigation backfired (-80% hit@10)** because of the
  anti-popularity gradient above: give the ranker tail candidates and it
  promotes them. Diagnosed with this notebook's own SHAP/LIME outputs.
- **Reserved-slot injection survives the diagnosis:** exploration items enter
  AFTER ranking (3 of 10 slots). Cost: hit@10 -13.8% relative
  (0.0146 -> 0.0126). Gains: coverage 0.071 -> 0.103, long-tail share
  0.010 -> 0.021, small-seller exposure x5, Gini 0.9986 -> 0.9921. This is the
  fairness-accuracy trade-off a marketplace can actually reason about.